## Real-Time Data Analytics in Modern Manufacturing

In today’s complex and rapidly changing manufacturing environment, the importance of **real-time data analytics** continues to grow. Specifically, the ability to process and analyze the vast amounts of data generated during production in real-time is critical for enhancing operational efficiency and quality control. At the heart of this process lies **Streaming Data Analytics**.

### Key Concepts of Streaming Data

Streaming data analysis refers to the technology used to process data immediately as it is continuously generated and transmitted. In manufacturing, it is essential to collect real-time data from various sources—such as **sensors, machinery, robots, and IoT devices**—to provide instantaneous feedback and predictive insights.

### Practical Applications

* **Real-Time Monitoring**: By monitoring live machine data, operators can immediately detect unexpected failures or operational inefficiencies.
* **Proactive Problem Solving**: Utilizing these technologies allows for issues to be resolved before they escalate, thereby maximizing overall productivity.
* **Instant Feedback**: Provides the foundation for automated adjustments on the factory floor, ensuring consistent quality.

## Scenario 1: Real-Time Streaming Data Analysis

The following is a Python example for streaming data analysis. This example covers the process of analyzing incoming real-time data and processing the results instantaneously. While libraries like **PySpark** are often used for large-scale streaming, this example demonstrates how to handle streaming data logic using **Pandas** for simplicity.

### Objective
In this scenario, we receive real-time data from **temperature and pressure sensors** and perform anomaly detection using a streaming approach. The system processes data one point at a time and calculates a **moving average** over a rolling window to detect deviations.


### Key Steps
1. **Real-time Ingestion**: Receiving individual data points sequentially (simulating a live feed).
2. **Windowed Processing**: Collecting data over a specific period to calculate metrics.
3. **Anomaly Detection**: Using the moving average to identify unexpected spikes or drops in sensor values.

In [5]:
import pandas as pd
import numpy as np
from collections import deque

# Function to generate synthetic streaming data
def generate_stream_data(n_samples=50):
    np.random.seed(42)
    data = {
        'Temperature': np.random.normal(75, 5, n_samples),
        'Pressure': np.random.normal(1000, 50, n_samples)
    }
    return pd.DataFrame(data)

# Function for streaming data analysis
def stream_data_analysis(df, window_size=10):
    # Deque to store the recent values for the rolling window
    temperature_window = deque(maxlen=window_size) 
    pressure_window = deque(maxlen=window_size) 

    for i in range(len(df)):
        current_temp = df.iloc[i]['Temperature']
        current_pressure = df.iloc[i]['Pressure']

        # Append new data point to the window
        temperature_window.append(current_temp)
        pressure_window.append(current_pressure)

        # Calculate moving average for Temperature
        temp_mean = np.mean(temperature_window)

        # Simple Anomaly Detection: Flag if Pressure exceeds 1050
        if current_pressure > 1050:
            pressure_status = "Anomaly"
        else:
            pressure_status = "Normal"

        # Output current status
        print(f"Data Batch #{i+1}")
        print(f"Current Temp: {current_temp:.2f}, Rolling Avg Temp ({window_size} samples): {temp_mean:.2f}")
        print(f"Current Pressure: {current_pressure:.2f}, Status: {pressure_status}\n")


# 1. Generate streaming data
stream_df = generate_stream_data()

# 2. Process and analyze streaming data
stream_data_analysis(stream_df, window_size=10)

Data Batch #1
Current Temp: 77.48, Rolling Avg Temp (10 samples): 77.48
Current Pressure: 1016.20, Status: Normal

Data Batch #2
Current Temp: 74.31, Rolling Avg Temp (10 samples): 75.90
Current Pressure: 980.75, Status: Normal

Data Batch #3
Current Temp: 78.24, Rolling Avg Temp (10 samples): 76.68
Current Pressure: 966.15, Status: Normal

Data Batch #4
Current Temp: 82.62, Rolling Avg Temp (10 samples): 78.16
Current Pressure: 1030.58, Status: Normal

Data Batch #5
Current Temp: 73.83, Rolling Avg Temp (10 samples): 77.30
Current Pressure: 1051.55, Status: Anomaly

Data Batch #6
Current Temp: 73.83, Rolling Avg Temp (10 samples): 76.72
Current Pressure: 1046.56, Status: Normal

Data Batch #7
Current Temp: 82.90, Rolling Avg Temp (10 samples): 77.60
Current Pressure: 958.04, Status: Normal

Data Batch #8
Current Temp: 78.84, Rolling Avg Temp (10 samples): 77.75
Current Pressure: 984.54, Status: Normal

Data Batch #9
Current Temp: 72.65, Rolling Avg Temp (10 samples): 77.19
Current Pre

## Result Analysis: Real-Time Streaming Data

### 1. Pressure Anomaly Detection
The system used a fixed threshold of **1050** to trigger real-time alerts.
* **Detection Events**: Anomalies were detected in Batches **#5, #16, #18, #22, #24, and #33**.
* **Observation**: The highest pressure recorded was **1078.23** (Batch #24). These spikes represent instantaneous risks that require immediate machine inspection or automated pressure release.



### 2. Temperature Trend Analysis (Rolling Average)
The **10-sample moving average** successfully smoothed out individual sensor fluctuations to show the underlying thermal trend.
* **Initial State**: The average stabilized around **77-78°C** during the first 10 batches.
* **Thermal Shift**: Between Batches #14 and #20, a noticeable drop in the rolling average occurred (dipping to **71.05°C**), suggesting a cooling phase in the process or a change in the environment.
* **Stability**: The temperature eventually stabilized back into the **73-74°C** range by the end of the stream (Batch #50).



### 3. Key Findings
| Metric | Value / Observation | Analysis |
| :--- | :--- | :--- |
| **Highest Pressure** | 1078.23 (Batch #24) | Critical anomaly; exceeded limit by 2.6%. |
| **Lowest Rolling Temp** | 71.05 (Batch #20) | Represents the lowest sustained thermal state. |
| **Detection Speed** | Instantaneous | The logic identified anomalies at the exact moment of ingestion. |

## Scenario 2: Real-Time Anomaly Detection using Isolation Forest

### Objective
The goal is to implement a machine learning-based model to monitor manufacturing sensors in real-time. By utilizing the **Isolation Forest** algorithm, the system identifies anomalies by isolating observations that are few and different. This approach is highly effective for detecting complex failure patterns in Temperature and Pressure data that simple thresholds might miss.



### Key Steps

1. **Data Collection & Scaling**: 
   Gather sensor data and apply **Standardization** (`StandardScaler`). This ensures that features with different scales are treated equally by the model.

2. **Model Training (Isolation Forest)**: 
   Train the Isolation Forest model on the scaled data. The `contamination` parameter is set (e.g., 0.05) to define the expected proportion of anomalies within the dataset.

3. **Real-Time Data Streaming Simulation**: 
   Process incoming sensor readings one row at a time to simulate a live manufacturing feed. Each sample is transformed using the pre-fitted scaler.

4. **Anomaly Prediction & Alerting**: 
   The model evaluates each sample and returns a prediction. A result of **-1** triggers an immediate **Anomaly Alert**, while **1** indicates **Normal** operation.

In [15]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Generate synthetic sensor data (Real-time streaming simulation)
def generate_sensor_data(n_samples=100):
    np.random.seed(42)
    data = {
        'Temperature': np.random.normal(75, 5, n_samples),
        'Pressure': np.random.normal(1000, 50, n_samples)
    }
    return pd.DataFrame(data)

# Real-time streaming data simulation function
def stream_sensor_data(df, model, scaler):
    for i in range(len(df)):
        sample = df.iloc[i:i+1] # Streaming data one row at a time
        sample_scaled = scaler.transform(sample) # Data scaling
        prediction = model.predict(sample_scaled) # Anomaly detection
        
        if prediction == -1:
            print(f"🚨 Anomaly detected! Data: {sample.values}")
        else:
            print(f"✅ Normal data: {sample.values}")

# 1. Sensor data collection
sensor_data = generate_sensor_data()

# 2. Data scaling
scaler = StandardScaler()
sensor_data_scaled = scaler.fit_transform(sensor_data)

# 3. Train anomaly detection model using Isolation Forest
# contamination=0.05 assumes a 5% expected outlier rate
isolation_forest = IsolationForest(contamination=0.05, random_state=42) 
isolation_forest.fit(sensor_data_scaled)

# 4. Real-time data streaming and anomaly detection
print("\nStarting real-time sensor data streaming... \n")
stream_sensor_data(sensor_data, isolation_forest, scaler)


Starting real-time sensor data streaming... 

✅ Normal data: [[ 77.48357077 929.2314629 ]]
✅ Normal data: [[ 74.30867849 978.96773386]]
✅ Normal data: [[ 78.23844269 982.86427417]]
✅ Normal data: [[ 82.61514928 959.88613654]]
✅ Normal data: [[ 73.82923313 991.93571442]]
✅ Normal data: [[  73.82931522 1020.20254284]]
🚨 Anomaly detected! Data: [[  82.89606408 1094.30929506]]
✅ Normal data: [[  78.83717365 1008.72889064]]
✅ Normal data: [[  72.65262807 1012.87751954]]
✅ Normal data: [[ 77.71280022 996.27770421]]
✅ Normal data: [[ 72.68291154 904.06143924]]
✅ Normal data: [[ 72.67135123 998.67430623]]
✅ Normal data: [[  76.20981136 1003.0115105 ]]
🚨 Anomaly detected! Data: [[  65.43359878 1123.16210562]]
✅ Normal data: [[ 66.37541084 990.38195176]]
✅ Normal data: [[  72.18856235 1015.07736712]]
✅ Normal data: [[ 69.9358444  998.26441151]]
✅ Normal data: [[ 76.57123666 941.56609812]]
✅ Normal data: [[  70.45987962 1057.14114073]]
✅ Normal data: [[  67.93848149 1037.59665163]]
✅ Normal data

## Result Analysis: Isolation Forest Anomaly Detection

### 1. Detection Summary
The Isolation Forest model processed 100 data points in a simulated real-time stream. With the `contamination` parameter set to 0.05, the model successfully isolated the most deviant samples from the collective group.

* **Total Anomalies Detected**: 5 events (Batches #7, #14, #32, #75, #80).
* **Detection Logic**: Unlike simple linear thresholds, the model flags points that are "easy to isolate", meaning they possess unusual combinations of Temperature and Pressure relative to the global distribution.

### 2. Analysis of Detected Anomalies
The detected anomalies generally fall into three categories of industrial concern:

| Batch # | Temperature | Pressure | Analysis |
| :--- | :--- | :--- | :--- |
| **#7** | 82.90 (High) | 1094.31 (High) | **Compound Risk**: Both sensors are significantly elevated. |
| **#14** | 65.43 (Low) | 1123.16 (High) | **Extreme Pressure**: Likely a pressure spike despite low heat. |
| **#32** | 84.26 (High) | 1003.43 (Norm) | **Thermal Outlier**: Excessive heat while pressure is stable. |
| **#75** | 61.90 (Low) | 1013.83 (Norm) | **System Drop**: Significant drop below normal operating temperature. |
| **#80** | 65.06 (Low) | 1136.01 (High) | **Critical Fault**: Maximum pressure recorded in the dataset. |

### 3. Model Performance Insights
* **Multivariate Awareness**: The model correctly identified Batch #7 as an anomaly because *both* variables were high, even though neither was at its absolute maximum.
* **Effectiveness**: The Isolation Forest approach proves more robust than fixed thresholds because it evaluates the **relationship** between sensors rather than just individual limits.